Install dependencies and load data

In [ ]:
!pip install transformers torch scikit-learn numpy pandas tqdm

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import normalize
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

In [ ]:
CONFIG = {
    'model_name': 'roberta-base',   
    'max_length': 512,            
    'embed_batch_size': 32,         
    'sent_per_round': 50,                   
    'total_rounds': 25,           
}

In [ ]:
labelled_df   = pd.read_csv("../../output/labelled/labelled.csv")
unlabelled_df = pd.read_csv("../../output/labelled/unlabelled.csv")
data          = pd.concat([labelled_df, unlabelled_df], ignore_index=True)
all_texts   = data["Sentence"].fillna("").astype(str).values
true_labels = data["Label"].values.copy()
labeled_positive_indices = list(np.where(true_labels == 1)[0])

Get sentence RoBERTa embeddings

In [ ]:
class RoBERTaClassifier(nn.Module):
    def __init__(self, model_name, dropout=0.1):
        super().__init__()
        self.roberta    = AutoModel.from_pretrained(model_name)
        hidden_size     = self.roberta.config.hidden_size
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 1)
        )

    def forward(self, input_ids, attention_mask):
        out     = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        cls     = out.last_hidden_state[:, 0, :]   # [CLS] token
        logit   = self.classifier(cls).squeeze(-1)
        return logit

    def get_embeddings(self, input_ids, attention_mask):
        with torch.no_grad():
            out = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        return out.last_hidden_state[:, 0, :].cpu().numpy()


print(f'Loading {CONFIG["model_name"]} ...')
tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_name'])
model     = RoBERTaClassifier(CONFIG['model_name']).to(DEVICE)
print('Model loaded.')

In [ ]:
def embed_all_documents(model, tokenizer, texts, config):
    """
    Embed every document in `texts` using the [CLS] token.
    Returns a normalised numpy array of shape (n_docs, hidden_size).
    """
    model.eval()
    all_embeddings = []
    batch_size     = config['embed_batch_size']

    for start in tqdm(range(0, len(texts), batch_size), desc='Embedding'):
        batch = [str(t) for t in texts[start : start + batch_size]]
        enc = tokenizer(
            batch,
            truncation=True,
            padding=True,
            max_length=config['max_length'],
            return_tensors='pt'
        ).to(DEVICE)
        embs  = model.get_embeddings(enc['input_ids'], enc['attention_mask'])
        all_embeddings.append(embs)

    embeddings = np.vstack(all_embeddings)
    embeddings = normalize(embeddings)   # L2 normalise for cosine similarity
    return embeddings


# Run the initial embedding pass
all_embeddings = embed_all_documents(model, tokenizer, all_texts, CONFIG)

Define helper functions for labelling loop

In [ ]:
def rank_by_classifier(clf, pool_indices, embeddings):
    """
    Rank pool by the classifier's predicted probability of being positive.
    """
    pool_vecs  = embeddings[pool_indices]
    scores     = clf.predict_proba(pool_vecs)[:, 1]
    order      = np.argsort(-scores)
    return [pool_indices[i] for i in order], scores[order]

def train_logistic_regression(Lp, Ln, embeddings):
    """
    Train a logistic regression model on the positive and negative labels
    """
    if len(Lp) == 0 or len(Ln) == 0:
        return None

    idx     = Lp + Ln
    X       = embeddings[idx]
    y       = [1]*len(Lp) + [0]*len(Ln)
    weights = [1.0]*len(Lp) + [1.0]*len(Ln)

    clf = LogisticRegression(
        class_weight='balanced',
        max_iter=1000,
        random_state=SEED
    )
    clf.fit(X, y, sample_weight=weights)
    return clf

In [ ]:
def label_top_k(ranked_pool, texts, k):
    """
    Show the top k documents to the user for labelling
    Returns two lists: new positives and new negatives.
    """
    candidates    = ranked_pool[:k]
    new_positives = []
    new_negatives = []

    print(f'\nPlease label the following {len(candidates)} documents.')
    print('Enter  y  for positive,  n  for negative\n')
    for i, idx in enumerate(candidates):
        print(f'--- Document {i+1}/{len(candidates)} (index {idx}) ---')
        print(texts[idx][:500])
        if len(texts[idx]) > 500:
            print('... [truncated]')
        while True:
            label = input('\nLabel (y/n): ').strip().lower()
            if label in ('y', 'n'):
                break
            print('Please enter y, n')
        if label == 'y':
            new_positives.append(idx)
        else:
            new_negatives.append(idx)
        print()
    return new_positives, new_negatives

Labelling loop

In [ ]:
Lp = list(labeled_positive_indices)
Ln = list(data[data["Label"] == 0].index)
labeled_set = set(Lp + Ln)
unlabeled_pool = [i for i in range(len(all_texts)) if i not in labeled_set]
clf = train_logistic_regression(Lp, Ln, all_embeddings)  # LR model

for round_num in range(1, CONFIG['total_rounds'] + 1):
    print(f'\n{"="*60}')
    print(f'ROUND {round_num}/{CONFIG["total_rounds"]}  |  '
          f'Lp={len(Lp)}  Ln={len(Ln)}  pool={len(unlabeled_pool)}')
    print('='*60)

    # Rank the unlabelled pool to determine which sentences to present to the user
    ranked_pool, scores = rank_by_classifier(clf, unlabeled_pool, all_embeddings)
    # Get the user to label the top k sentences
    new_pos, new_neg = label_top_k(ranked_pool, all_texts, k=CONFIG["sent_per_round"])
    print(f'  Labeled: +{len(new_pos)} positives, -{len(new_neg)} negatives')

    # Update sets and retrain LR
    Lp += new_pos
    Ln += new_neg
    labeled_this_round = set(new_pos + new_neg)
    unlabeled_pool     = [i for i in unlabeled_pool if i not in labeled_this_round]
    clf = train_logistic_regression(Lp, Ln, all_embeddings)

    for idx in new_pos:
        data.at[idx, "Label"] = 1
    for idx in new_neg:
        data.at[idx, "Label"] = 0

In [ ]:
# Save labels
labelled_df = data[data["Label"] != -1].copy()
unlabelled_df = data[data["Label"] == -1].copy()
labelled_df.to_csv("../../output/labelled/labelled.csv", index=False)
unlabelled_df.to_csv("../../output/labelled/unlabelled.csv", index=False)